# 🧪 MolForge GRPO Training — `environment_factory` Pattern

This notebook trains a language model to design drug candidates using the **MolForge** environment
via TRL's official `environment_factory` GRPO pattern — the same approach used in the
[Sudoku](https://github.com/huggingface/trl/blob/main/examples/notebooks/openenv_sudoku_grpo.ipynb)
and [Wordle](https://github.com/huggingface/trl/blob/main/examples/scripts/openenv/wordle.py) examples.

**How it works:**
1. The model learns to call **tool functions** (`edit_molecule`, `run_assay`, `submit_molecule`, etc.)
2. The `GRPOTrainer` automatically manages multi-turn rollouts via the environment
3. GRPO reinforcement learning optimizes the model based on environment reward signals

**Requirements:** Google Colab with T4 GPU

## 1. Install Dependencies

In [ ]:
%%capture
# Install TRL with vLLM support and all dependencies
!pip install -U "trl[vllm]" pydantic datasets matplotlib
# Optional: Unsloth for 2x faster training
# !pip install -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Clone MolForge Repo

In [ ]:
import os
import sys

REPO_URL = "https://github.com/Adhitya-Vardhan/molt_lab.git"
PROJECT_ROOT = "/content/molt_lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone {REPO_URL} {PROJECT_ROOT}

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, "issue"))

# Bootstrap openenv shim (replaces heavy openenv-core for training)
try:
    import openenv
except ImportError:
    import openenv_shim

# Set reward mode for training
os.environ["MOLFORGE_REWARD_MODE"] = "curriculum"  # warm-up mode
os.environ["MOLFORGE_TRAINING_RANDOMIZATION"] = "1"

print(f"✅ Project ready at {PROJECT_ROOT}")

## 3. Verify Environment

In [ ]:
from molforge_tool_env import MolForgeToolEnv

# Quick smoke test
env = MolForgeToolEnv()
obs_text = env.reset()
print("=== Initial Observation ===")
print(obs_text)
print()

# Test a tool call
result = env.run_assay(
    tool_name="evaluate_properties",
    rationale="Check baseline properties of starting molecule."
)
print("=== After run_assay ===")
print(result)
print(f"\nReward so far: {env.reward}")
print(f"Done: {env.done}")

## 4. Define System Prompt & Dataset

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are an expert medicinal chemist designing kinase inhibitor drug candidates.

## YOUR TASK

Design a molecule that meets the scenario's potency, toxicity, and synthesizability constraints
within a limited oracle budget. You work by editing a modular molecule (4 structural slots) and
running computational assays to gather evidence before submitting your final candidate.

## AVAILABLE TOOLS

You have 5 tools to interact with the environment:

1. **edit_molecule(slot, fragment, edit_type, rationale)** — Edit a molecular slot.
   - slot: warhead, hinge, solvent_tail, or back_pocket
   - fragment: the fragment identifier to place
   - edit_type: add_fragment, substitute, remove, or undo_last_edit
   - rationale: why this edit should improve the molecule

2. **run_assay(tool_name, rationale)** — Run a computational assay.
   - tool_name: evaluate_properties, dock_target, assay_toxicity, estimate_synthesizability,
     evaluate_novelty, search_literature, or run_md_simulation
   - rationale: why this assay is needed now

3. **submit_molecule(rationale)** — Submit the current molecule as the final candidate.
   - Only submit when you have sufficient evidence that constraints are met.

4. **restart_episode(rationale)** — Restart with a fresh scaffold.
   - Use when the current molecular series is fundamentally flawed.

5. **defer_action(rationale)** — Skip this turn.
   - Use when no productive action is available.

## STRATEGY

1. First, run assays to understand the starting molecule's properties.
2. Make targeted edits to improve weak properties.
3. Re-run assays to verify improvements.
4. Submit when evidence shows all constraints are satisfied.
5. Be budget-conscious — each assay costs oracle budget points.
"""

DATASET_SIZE = 200  # Number of training episodes

dataset = Dataset.from_dict({
    "prompt": [[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Design a drug candidate that meets the scenario constraints. Use the available tools to edit the molecule, run assays, and submit when ready."},
    ]] * DATASET_SIZE
})

print(f"✅ Dataset: {len(dataset)} training episodes")

## 5. Define Reward Functions

In [ ]:
from molforge_tool_env import (
    reward_environment,
    reward_valid_actions,
    reward_progress,
    reward_submission,
)

reward_funcs = [
    reward_environment,     # Main env reward (sum of step rewards)
    reward_valid_actions,   # Valid action ratio
    reward_progress,        # Progress toward solving scenario
    reward_submission,      # Binary: did the model submit?
]

print("✅ Reward functions:")
for fn in reward_funcs:
    print(f"  - {fn.__name__}: {fn.__doc__.strip()}")

## 6. Configure & Train

In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer

MODEL_NAME = "Qwen/Qwen3-0.6B"  # Small model for T4; use Qwen3-1.7B on A100

grpo_config = GRPOConfig(
    # Training schedule
    num_train_epochs=1,
    learning_rate=5e-6,
    gradient_accumulation_steps=16,
    per_device_train_batch_size=1,
    warmup_steps=10,
    optim="adamw_torch",
    max_grad_norm=1.0,
    max_steps=50,

    # GRPO
    num_generations=2,
    max_completion_length=4096,
    log_completions=True,
    num_completions_to_print=1,
    chat_template_kwargs={"enable_thinking": False},

    # vLLM for fast generation on Colab T4
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.15,

    # Logging
    output_dir="molforge-grpo-output",
    logging_steps=1,
    save_steps=10,
    save_total_limit=2,

    # Sampling
    temperature=0.8,
    top_k=10,

    # Seed
    seed=3407,
)

In [ ]:
trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=reward_funcs,
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=MolForgeToolEnv,
)

if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
    print(f"{start_gpu_memory} GB of memory reserved.")

print("\n🚀 Starting GRPO training with MolForge environment...")

In [ ]:
trainer.train()

## 7. Save Results

In [ ]:
# Save the trained model
trainer.save_model("molforge-grpo-output")
print("✅ Model saved to molforge-grpo-output")

# Optionally push to Hub
# trainer.push_to_hub()

# GPU stats
if torch.cuda.is_available():
    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    print(f"Peak GPU memory: {used_memory} GB")

print("\n🎉 MolForge GRPO training complete!")

## 8. (Optional) Copy to Google Drive

In [ ]:
import shutil
from pathlib import Path

# Mount drive if available
try:
    from google.colab import drive
    drive.mount("/content/drive")
    
    DRIVE_OUTPUT = "/content/drive/MyDrive/MolForge_RL_Runs/latest"
    Path(DRIVE_OUTPUT).parent.mkdir(parents=True, exist_ok=True)
    
    if Path(DRIVE_OUTPUT).exists():
        shutil.rmtree(DRIVE_OUTPUT)
    shutil.copytree("molforge-grpo-output", DRIVE_OUTPUT)
    print(f"✅ Copied outputs to {DRIVE_OUTPUT}")
except Exception as e:
    print(f"⚠️ Could not copy to Drive: {e}")